# Experiment 2: ALP-GMM Automatic Curriculum for Multi-Floor Maze

## Phương pháp: Absolute Learning Progress — Gaussian Mixture Model

Thay vì curriculum cố định 9 level tuần tự (Experiment 1), ta dùng **ALP-GMM** (Portelas et al., 2019) để **tự động tạo curriculum**.

### Ý tưởng cốt lõi
1. **Không gian tham số liên tục**: Mỗi task = vector 7 chiều `(grid_size, n_floors, n_traps, n_locks, n_enemies, enemy_mix, agent_hp)`
2. **Absolute Learning Progress (ALP)**: Đo tiến bộ = `|reward_mới − reward_task_gần_nhất|`
3. **GMM Sampling**: Fit GMM lên phân phối ALP → sample task ở vùng agent đang tiến bộ **nhiều nhất**
4. **ε-greedy**: Cân bằng exploration (random task) vs exploitation (task từ GMM)

### Lợi thế so với curriculum cố định
- Agent tự tìm "biên giới học tập" tối ưu
- Không lãng phí thời gian ở task đã master
- Anti-forgetting tự nhiên: tự quay lại vùng đang suy giảm

### Reference
- Portelas et al., *"Teacher algorithms for curriculum learning of Deep RL in continuously parameterized environments"*, CoRL 2019
- [Curriculum for RL — Lil'Log](https://lilianweng.github.io/posts/2020-01-29-curriculum-rl/)

In [ ]:
from pathlib import Path
import importlib
import subprocess
import sys

REQUIRED_MODULES = {
    "gymnasium": "gymnasium",
    "stable_baselines3": "stable-baselines3",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}

missing = []
for mod, pkg in REQUIRED_MODULES.items():
    try:
        importlib.import_module(mod)
    except ImportError:
        missing.append(pkg)

if missing:
    print("Installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All packages ready.")

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

if not (ROOT / "mfm" / "env.py").exists():
    raise FileNotFoundError(
        "mfm/env.py not found. If running on Kaggle, copy the bootstrap cell "
        "from RL_Compare_All_Models_Kaggle.ipynb to embed env.py first."
    )

print("Root:", ROOT)

In [ ]:
import copy
import gc
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from gymnasium import spaces
from scipy.spatial import cKDTree
from sklearn.mixture import GaussianMixture
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from mfm.env import CURRICULUM, MazeEnv

# ── Cấu hình chung ──────────────────────────────────────────
DEVICE = "cpu"
N_ENVS = 4
EVAL_EPISODES = 30
SEED = 42

TOTAL_TIMESTEPS = 3_500_000
STEPS_PER_TASK = 20_000
TIMESTEP_SCALE = 1.0  # 0.1 hoặc 0.25 để smoke-test

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Total timesteps: {int(TOTAL_TIMESTEPS * TIMESTEP_SCALE):,}")
print(f"Steps per task: {STEPS_PER_TASK:,}")

## ALP-GMM: Teacher tự động

**Thuật toán:**
1. Ban đầu: sample task ngẫu nhiên (warm-up phase)
2. Sau khi thu đủ mẫu → fit GMM lên không gian `(params, ALP)`
3. Sample mới: ε-greedy — random hoặc từ GMM (vùng ALP cao = đang tiến bộ)
4. KD-Tree để tìm task gần nhất nhanh (nearest neighbor cho tính ALP)

In [ ]:
class ALPGMM:
    """Absolute Learning Progress — Gaussian Mixture Model.

    Portelas et al. (2019): Teacher algorithms for curriculum learning
    of Deep RL in continuously parameterized environments.
    """

    def __init__(
        self,
        param_bounds,
        n_components=5,
        random_ratio=0.2,
        fit_rate=25,
        min_samples=50,
        window_size=500,
        seed=42,
    ):
        self.bounds = np.array(param_bounds, dtype=np.float64)
        self.n_dims = len(param_bounds)
        self.n_components = n_components
        self.random_ratio = random_ratio
        self.fit_rate = fit_rate
        self.min_samples = min_samples
        self.window_size = window_size
        self.rng = np.random.RandomState(seed)

        self.all_params = []
        self.all_rewards = []
        self.all_alp = []
        self.gmm = None
        self._kdtree = None

    # ── Normalization ────────────────────────────────────────
    def _normalize(self, params):
        lo, hi = self.bounds[:, 0], self.bounds[:, 1]
        span = hi - lo
        span[span == 0] = 1.0
        return (np.asarray(params) - lo) / span

    def _denormalize(self, normed):
        lo, hi = self.bounds[:, 0], self.bounds[:, 1]
        return np.asarray(normed) * (hi - lo) + lo

    # ── ALP computation ──────────────────────────────────────
    def _compute_alp(self, params, reward):
        if len(self.all_params) < 2:
            return abs(reward)
        norm_p = self._normalize(params).reshape(1, -1)
        _, idx = self._kdtree.query(norm_p, k=1)
        return abs(reward - self.all_rewards[int(idx)])

    # ── Update after each task ───────────────────────────────
    def update(self, params, reward):
        alp = self._compute_alp(params, reward)

        self.all_params.append(np.array(params, dtype=np.float64))
        self.all_rewards.append(float(reward))
        self.all_alp.append(float(alp))

        norm_hist = np.array([self._normalize(p) for p in self.all_params])
        self._kdtree = cKDTree(norm_hist)

        n = len(self.all_params)
        if n >= self.min_samples and n % self.fit_rate == 0:
            self._fit_gmm()

        return alp

    # ── Fit GMM (weighted by ALP) ────────────────────────────
    def _fit_gmm(self):
        w = self.window_size
        params_arr = np.array(self.all_params[-w:])
        alp_arr = np.array(self.all_alp[-w:])
        norm_params = np.array([self._normalize(p) for p in params_arr])

        weights = alp_arr + 1e-8
        weights /= weights.sum()

        n_comp = min(self.n_components, max(2, len(params_arr) // 8))
        oversample = min(3, max(1, 200 // len(params_arr) + 1))
        indices = self.rng.choice(
            len(params_arr),
            size=len(params_arr) * oversample,
            p=weights,
            replace=True,
        )
        weighted_data = norm_params[indices]

        try:
            gmm = GaussianMixture(
                n_components=n_comp,
                covariance_type="full",
                max_iter=150,
                random_state=self.rng.randint(100000),
            )
            gmm.fit(weighted_data)
            self.gmm = gmm
        except Exception as e:
            print(f"  [ALP-GMM] GMM fit failed: {e}")

    # ── Sample next task ─────────────────────────────────────
    def sample(self):
        if self.gmm is None or self.rng.random() < self.random_ratio:
            return np.array([
                self.rng.uniform(lo, hi)
                for lo, hi in self.bounds
            ])

        raw, _ = self.gmm.sample(1)
        normed = np.clip(raw[0], 0.0, 1.0)
        params = self._denormalize(normed)
        return np.clip(params, self.bounds[:, 0], self.bounds[:, 1])

    # ── Diagnostics ──────────────────────────────────────────
    def stats(self):
        if not self.all_alp:
            return {}
        recent = self.all_alp[-50:]
        return {
            "total_tasks": len(self.all_params),
            "mean_alp_recent": float(np.mean(recent)),
            "max_alp_recent": float(np.max(recent)),
            "gmm_fitted": self.gmm is not None,
            "gmm_components": self.gmm.n_components if self.gmm else 0,
        }


print("ALP-GMM class ready.")

## Không gian tham số Task

7 chiều liên tục, mỗi chiều map sang 1 tham số cấu hình maze:

| Dim | Tham số | Range | Ý nghĩa |
|-----|---------|-------|----------|
| 0 | `grid_size` | [7, 13] | Kích thước grid (luôn lẻ) |
| 1 | `n_floors` | [1, 3] | Số tầng |
| 2 | `n_traps` | [0, 8] | Số bẫy |
| 3 | `n_locks` | [0, 3] | Số cặp key-door |
| 4 | `n_enemies` | [0, 4] | Tổng số enemy |
| 5 | `enemy_mix` | [0, 1] | Loại enemy: 0=patrol, 0.5=+chaser, 1=+sniper |
| 6 | `agent_hp` | [5, 25] | Máu agent (thấp = khó hơn) |

In [ ]:
PARAM_BOUNDS = [
    (7.0,  13.0),   # 0: grid_size
    (1.0,   3.0),   # 1: n_floors
    (0.0,   8.0),   # 2: n_traps
    (0.0,   3.0),   # 3: n_locks
    (0.0,   4.0),   # 4: n_enemies
    (0.0,   1.0),   # 5: enemy_mix
    (5.0,  25.0),   # 6: agent_hp
]
PARAM_NAMES = [
    "grid_size", "n_floors", "n_traps", "n_locks",
    "n_enemies", "enemy_mix", "agent_hp",
]


def params_to_config(params):
    """Chuyển vector 7-dim liên tục thành dict config cho MazeEnv."""
    grid = int(np.clip(round(params[0]), 7, 13))
    if grid % 2 == 0:
        grid = grid + 1 if grid < 13 else grid - 1
    n_floors = int(np.clip(round(params[1]), 1, 3))
    n_traps  = int(np.clip(round(params[2]), 0, 8))
    n_locks  = int(np.clip(round(params[3]), 0, min(3, n_floors)))
    n_enemies = int(np.clip(round(params[4]), 0, 4))
    enemy_mix = float(np.clip(params[5], 0, 1))
    agent_hp  = int(np.clip(round(params[6]), 5, 25))

    enemies = []
    for i in range(n_enemies):
        fl = i % n_floors
        if enemy_mix >= 0.7 and i == 0:
            etype = "sniper"
        elif enemy_mix >= 0.35 and i % 2 == 0:
            etype = "chaser"
        else:
            etype = "patrol"
        enemies.append({"type": etype, "floor": fl})

    locks = [{"k": i % n_floors, "d": i % n_floors} for i in range(n_locks)]

    base_steps = grid * grid
    factor = (1 + 0.5 * (n_floors - 1)) * (1 + 0.15 * n_enemies) * (1 + 0.2 * n_locks)
    max_steps = int(np.clip(base_steps * factor, 50, 500))

    agent_ammo = min(n_enemies * 3, 8) if n_enemies > 0 else 0

    return {
        "name": f"G{grid}_F{n_floors}_T{n_traps}_L{n_locks}_E{n_enemies}",
        "width": grid, "height": grid,
        "n_floors": n_floors,
        "max_steps": max_steps,
        "agent_hp": agent_hp,
        "agent_ammo": agent_ammo,
        "agent_max_ammo": agent_ammo,
        "agent_stamina": 3 + n_floors,
        "n_traps": n_traps,
        "n_health": max(2, n_traps // 2 + n_enemies),
        "n_ammo": max(0, n_enemies),
        "enemies": enemies,
        "locks": locks,
        "start_floor": n_floors - 1 if n_floors > 1 else 0,
        "goal_floor": 0,
    }


def difficulty_score(params):
    """Scalar [0, 1] đo độ khó tổng hợp — dùng cho visualization."""
    ws = np.array([0.15, 0.20, 0.10, 0.15, 0.20, 0.10, 0.10])
    norms = np.array([
        (params[0] - 7) / 6,
        (params[1] - 1) / 2,
        params[2] / 8,
        params[3] / 3,
        params[4] / 4,
        params[5],
        1.0 - (params[6] - 5) / 20,  # thấp HP → khó hơn
    ])
    return float(np.dot(ws, np.clip(norms, 0, 1)))


cfg_test = params_to_config(np.array([11, 2, 3, 1, 2, 0.5, 15]))
print("Sample config:", json.dumps(cfg_test, indent=2))
print("Difficulty:", f"{difficulty_score(np.array([11, 2, 3, 1, 2, 0.5, 15])):.3f}")

## Model & Utilities

Dùng cùng kiến trúc `SmallDictExtractor` + PPO như Experiment 1 để so sánh công bằng.
Chỉ thay đổi **cách chọn task** (ALP-GMM thay vì tuần tự).

In [ ]:
class SmallDictExtractor(BaseFeaturesExtractor):
    def __init__(self, observation_space: spaces.Dict, features_dim: int = 128):
        super().__init__(observation_space, features_dim)
        n_ch = observation_space["visual"].shape[0]
        vec_dim = observation_space["vector"].shape[0]
        self.cnn = nn.Sequential(
            nn.Conv2d(n_ch, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
        )
        self.vec_net = nn.Sequential(
            nn.Linear(vec_dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
        )
        self.fusion = nn.Sequential(nn.Linear(128, features_dim), nn.ReLU())

    def forward(self, obs):
        vis = self.cnn(obs["visual"])
        vec = self.vec_net(obs["vector"])
        return self.fusion(torch.cat([vis, vec], dim=1))


POLICY_KWARGS = dict(
    features_extractor_class=SmallDictExtractor,
    features_extractor_kwargs=dict(features_dim=128),
)


def make_env(level_config, seed=None):
    def _init():
        env = MazeEnv(config=copy.deepcopy(level_config))
        if seed is not None:
            env.reset(seed=seed)
        return env
    return _init


def make_vec_env(level_config, n_envs=N_ENVS):
    envs = DummyVecEnv([make_env(level_config, seed=SEED + i * 1000) for i in range(n_envs)])
    return VecNormalize(envs, norm_obs=False, norm_reward=True, clip_reward=10.0, gamma=0.99)


def build_ppo(env):
    return PPO(
        "MultiInputPolicy", env,
        learning_rate=3e-4, n_steps=256, batch_size=128, n_epochs=4,
        gamma=0.99, gae_lambda=0.95, clip_range=0.2,
        ent_coef=0.01, vf_coef=0.5, max_grad_norm=0.5,
        policy_kwargs={
            **POLICY_KWARGS,
            "net_arch": dict(pi=[128, 64], vf=[128, 64]),
            "activation_fn": nn.ReLU,
        },
        device=DEVICE, verbose=0,
    )


class TrainLogger(BaseCallback):
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.ep_rewards, self.ep_wins = [], []
        self._running = None

    def _on_training_start(self):
        self._running = np.zeros(self.training_env.num_envs)

    def _on_step(self):
        rews = self.locals.get("rewards")
        dones = self.locals.get("dones")
        infos = self.locals.get("infos", [])
        if rews is None or dones is None:
            return True
        if np.isscalar(rews):
            rews, dones = np.array([rews]), np.array([dones])
        if self._running is None:
            self._running = np.zeros(len(rews))
        self._running[:len(rews)] += rews
        for i, d in enumerate(dones):
            if d:
                self.ep_rewards.append(float(self._running[i]))
                self._running[i] = 0.0
                if i < len(infos) and "win" in infos[i]:
                    self.ep_wins.append(1.0 if infos[i]["win"] else 0.0)
        return True


def evaluate(model, level_config, n_episodes=10):
    rewards, wins = [], []
    for s in range(9000, 9000 + n_episodes):
        env = MazeEnv(config=copy.deepcopy(level_config))
        obs, _ = env.reset(seed=s)
        done, total_r = False, 0.0
        while not done:
            action, _ = model.predict(obs, deterministic=True)
            obs, r, term, trunc, info = env.step(int(action))
            total_r += r
            done = term or trunc
        rewards.append(total_r)
        wins.append(1.0 if info.get("win", False) else 0.0)
    return float(np.mean(rewards)), float(np.mean(wins))


print("Model utilities ready.")

## Training Loop — ALP-GMM Curriculum

Vòng lặp chính:
1. ALP-GMM sample task params → tạo MazeEnv config
2. Train PPO trên task đó `STEPS_PER_TASK` bước
3. Evaluate → lấy reward
4. Cập nhật ALP-GMM (tính ALP, rebuild KD-Tree, refit GMM)
5. Lặp lại cho đến `TOTAL_TIMESTEPS`

Mỗi `CHECKPOINT_INTERVAL` bước: eval trên 9 level gốc + save model.

In [ ]:
CHECKPOINT_INTERVAL = 500_000  # eval trên 9 level gốc mỗi 500K steps

scaled_total = int(TOTAL_TIMESTEPS * TIMESTEP_SCALE)

alp_gmm = ALPGMM(
    param_bounds=PARAM_BOUNDS,
    n_components=5,
    random_ratio=0.20,
    fit_rate=25,
    min_samples=40,
    window_size=400,
    seed=SEED,
)

# ── Khởi tạo model với task đơn giản nhất ──────────────────
init_params = np.array([7.0, 1.0, 0.0, 0.0, 0.0, 0.0, 20.0])
init_cfg = params_to_config(init_params)
vec_env = make_vec_env(init_cfg)
model = build_ppo(vec_env)

total_params = sum(p.numel() for p in model.policy.parameters())
print(f"PPO params: {total_params:,}")

# ── Logging ─────────────────────────────────────────────────
log = {
    "task_steps": [],
    "task_rewards": [],
    "task_wins": [],
    "task_difficulties": [],
    "task_alp": [],
    "task_params": [],
    "task_names": [],
    "checkpoint_steps": [],
    "checkpoint_level_rewards": [],
    "checkpoint_level_wins": [],
}

cumulative_steps = 0
task_count = 0
last_checkpoint = 0
t0_total = time.time()

print(f"\n{'=' * 70}")
print(f"ALP-GMM Training — {scaled_total:,} total timesteps")
print(f"{'=' * 70}\n")

while cumulative_steps < scaled_total:
    params = alp_gmm.sample()
    cfg = params_to_config(params)
    diff = difficulty_score(params)

    vec_env.close()
    vec_env = make_vec_env(cfg)
    model.set_env(vec_env)

    cb = TrainLogger()
    model.learn(total_timesteps=STEPS_PER_TASK, callback=cb, reset_num_timesteps=False)
    cumulative_steps += STEPS_PER_TASK
    task_count += 1

    mean_r, win_rate = evaluate(model, cfg, n_episodes=10)
    alp = alp_gmm.update(params, mean_r)

    log["task_steps"].append(cumulative_steps)
    log["task_rewards"].append(round(mean_r, 2))
    log["task_wins"].append(round(win_rate, 4))
    log["task_difficulties"].append(round(diff, 3))
    log["task_alp"].append(round(alp, 4))
    log["task_params"].append(params.tolist())
    log["task_names"].append(cfg["name"])

    if task_count % 5 == 0 or cumulative_steps >= scaled_total:
        stats = alp_gmm.stats()
        print(
            f"Task {task_count:>4d} | steps={cumulative_steps:>9,d} | "
            f"diff={diff:.2f} | R={mean_r:>7.2f} | win={win_rate:.0%} | "
            f"ALP={alp:.3f} | GMM={'Y' if stats.get('gmm_fitted') else 'N'} | "
            f"{cfg['name']}"
        )

    if cumulative_steps - last_checkpoint >= CHECKPOINT_INTERVAL or cumulative_steps >= scaled_total:
        last_checkpoint = cumulative_steps
        print(f"\n  ── Checkpoint @ {cumulative_steps:,} steps ──")
        ckpt_rewards, ckpt_wins = {}, {}
        for lv in range(1, 10):
            lv_cfg = copy.deepcopy(CURRICULUM[lv])
            mr, wr = evaluate(model, lv_cfg, n_episodes=EVAL_EPISODES)
            ckpt_rewards[lv] = round(mr, 2)
            ckpt_wins[lv] = round(wr, 4)
            print(f"    Level {lv} ({lv_cfg['name']:<12s}) | R={mr:>7.2f} | win={wr:.1%}")
        log["checkpoint_steps"].append(cumulative_steps)
        log["checkpoint_level_rewards"].append(ckpt_rewards)
        log["checkpoint_level_wins"].append(ckpt_wins)

        model.save(str(OUTPUT_DIR / f"alp_gmm_ppo_ckpt_{cumulative_steps}.zip"))
        print()

elapsed = time.time() - t0_total
print(f"\nTotal training time: {elapsed/60:.1f} min")

model.save(str(OUTPUT_DIR / "alp_gmm_ppo_final.zip"))
print("Final model saved.")

vec_env.close()

## Final Evaluation & Lưu kết quả

Eval model cuối trên cả 9 level gốc (giống Experiment 1) để so sánh generalization.

In [ ]:
print("=" * 60)
print("FINAL EVALUATION — ALP-GMM PPO on all 9 original levels")
print("=" * 60)

final_results = {}
for lv in range(1, 10):
    lv_cfg = copy.deepcopy(CURRICULUM[lv])
    mr, wr = evaluate(model, lv_cfg, n_episodes=EVAL_EPISODES)
    final_results[lv] = {"name": lv_cfg["name"], "reward": round(mr, 2), "success": round(wr, 4)}
    print(f"  Level {lv:>2d} ({lv_cfg['name']:<12s}) | reward={mr:>7.2f} | success={wr:.1%}")

# ── Unseen task generalization test ──────────────────────────
print("\nUnseen task generalization (random configs not in training):")
unseen_rng = np.random.RandomState(7777)
unseen_results = []
for i in range(20):
    p = np.array([unseen_rng.uniform(lo, hi) for lo, hi in PARAM_BOUNDS])
    cfg = params_to_config(p)
    mr, wr = evaluate(model, cfg, n_episodes=10)
    d = difficulty_score(p)
    unseen_results.append({"diff": d, "reward": mr, "win": wr, "name": cfg["name"]})

unseen_results.sort(key=lambda x: x["diff"])
for r in unseen_results:
    print(f"  diff={r['diff']:.2f} | R={r['reward']:>7.2f} | win={r['win']:.0%} | {r['name']}")

# ── Save all results ─────────────────────────────────────────
results_all = {
    "method": "ALP-GMM",
    "total_timesteps": cumulative_steps,
    "total_tasks": task_count,
    "training_time_min": round(elapsed / 60, 1),
    "total_params": total_params,
    "final_per_level": {str(k): v for k, v in final_results.items()},
    "unseen_generalization": unseen_results,
    "alp_gmm_stats": alp_gmm.stats(),
    "log": log,
}

results_path = OUTPUT_DIR / "alp_gmm_results.json"
results_path.write_text(json.dumps(results_all, indent=2), encoding="utf-8")
print(f"\nResults saved: {results_path}")

## Visualization

4 biểu đồ chính:
1. **Task Difficulty Over Time** — ALP-GMM tự điều chỉnh độ khó thế nào?
2. **ALP Over Time** — Learning progress đang tập trung ở đâu?
3. **Task Parameter Heatmap** — Vùng nào của parameter space được explore nhiều?
4. **Final Performance on 9 Levels** — So sánh với Experiment 1

In [ ]:
def smooth(arr, window=15):
    if len(arr) < window:
        return arr
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode="valid")


fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("ALP-GMM Curriculum Analysis", fontsize=16, fontweight="bold")

steps = np.array(log["task_steps"])
diffs = np.array(log["task_difficulties"])
alps = np.array(log["task_alp"])
rews = np.array(log["task_rewards"])
wins = np.array(log["task_wins"])

# ── 1. Difficulty over time ──────────────────────────────────
ax = axes[0, 0]
ax.scatter(steps / 1e6, diffs, alpha=0.15, s=8, c="steelblue", label="Per task")
if len(diffs) >= 15:
    s_steps = steps[:len(smooth(diffs))]
    ax.plot(s_steps / 1e6, smooth(diffs), color="navy", linewidth=2, label="Smoothed")
ax.set_xlabel("Timesteps (M)")
ax.set_ylabel("Task Difficulty")
ax.set_title("Task Difficulty Over Training")
ax.set_ylim(-0.05, 1.05)
ax.legend()
ax.grid(alpha=0.3)

# ── 2. ALP over time ────────────────────────────────────────
ax = axes[0, 1]
ax.scatter(steps / 1e6, alps, alpha=0.15, s=8, c="coral", label="Per task")
if len(alps) >= 15:
    s_steps2 = steps[:len(smooth(alps))]
    ax.plot(s_steps2 / 1e6, smooth(alps), color="darkred", linewidth=2, label="Smoothed")
ax.set_xlabel("Timesteps (M)")
ax.set_ylabel("ALP")
ax.set_title("Absolute Learning Progress")
ax.legend()
ax.grid(alpha=0.3)

# ── 3. Parameter exploration heatmap ────────────────────────
ax = axes[1, 0]
params_arr = np.array(log["task_params"])
if len(params_arr) > 0:
    normed = np.zeros_like(params_arr)
    for j in range(params_arr.shape[1]):
        lo, hi = PARAM_BOUNDS[j]
        span = hi - lo if hi != lo else 1.0
        normed[:, j] = (params_arr[:, j] - lo) / span

    n_bins = min(len(params_arr), 10)
    chunk_size = len(params_arr) // n_bins
    heatmap = np.zeros((7, n_bins))
    for b in range(n_bins):
        sl = normed[b * chunk_size:(b + 1) * chunk_size]
        heatmap[:, b] = sl.mean(axis=0)

    im = ax.imshow(heatmap, aspect="auto", cmap="YlOrRd", vmin=0, vmax=1)
    ax.set_yticks(range(7))
    ax.set_yticklabels(PARAM_NAMES, fontsize=9)
    ax.set_xlabel("Training Progress (bins)")
    ax.set_title("Parameter Exploration Heatmap")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

# ── 4. Final performance on 9 levels ────────────────────────
ax = axes[1, 1]
levels = list(range(1, 10))
level_names = [CURRICULUM[lv]["name"] for lv in levels]
success_rates = [final_results[lv]["success"] * 100 for lv in levels]
rewards_lv = [final_results[lv]["reward"] for lv in levels]

x = np.arange(len(levels))
bars = ax.bar(x, success_rates, color="mediumseagreen", edgecolor="darkgreen", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(level_names, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Success Rate (%)")
ax.set_ylim(0, 110)
ax.set_title("ALP-GMM Final Model — 9 Original Levels")
ax.grid(axis="y", alpha=0.3)

for bar, sr in zip(bars, success_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
            f"{sr:.0f}%", ha="center", va="bottom", fontsize=8, fontweight="bold")

plt.tight_layout()
chart_path = OUTPUT_DIR / "alp_gmm_curriculum_analysis.png"
fig.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path}")

In [ ]:
# ── Biểu đồ bổ sung: Reward & Win rate theo thời gian ────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle("ALP-GMM Training Progress", fontsize=14, fontweight="bold")

ax = axes[0]
ax.scatter(steps / 1e6, rews, alpha=0.12, s=6, c="royalblue")
if len(rews) >= 15:
    s_r = smooth(rews)
    ax.plot(steps[:len(s_r)] / 1e6, s_r, color="darkblue", linewidth=2)
ax.set_xlabel("Timesteps (M)")
ax.set_ylabel("Mean Reward")
ax.set_title("Task Reward Over Training")
ax.grid(alpha=0.3)

ax = axes[1]
ax.scatter(steps / 1e6, wins * 100, alpha=0.12, s=6, c="orange")
if len(wins) >= 15:
    s_w = smooth(wins * 100)
    ax.plot(steps[:len(s_w)] / 1e6, s_w, color="darkorange", linewidth=2)
ax.set_xlabel("Timesteps (M)")
ax.set_ylabel("Win Rate (%)")
ax.set_title("Task Win Rate Over Training")
ax.set_ylim(-5, 105)
ax.grid(alpha=0.3)

plt.tight_layout()
chart_path2 = OUTPUT_DIR / "alp_gmm_training_progress.png"
fig.savefig(chart_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {chart_path2}")

In [ ]:
# ── Checkpoint progression: performance on 9 levels over training ─
if len(log["checkpoint_steps"]) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle("Generalization Over Training", fontsize=14, fontweight="bold")

    ckpt_steps = log["checkpoint_steps"]
    colors_lv = plt.cm.viridis(np.linspace(0.1, 0.9, 9))

    for panel, metric, ylabel, title in [
        (0, "checkpoint_level_wins", "Success Rate (%)", "Success Rate on Original Levels"),
        (1, "checkpoint_level_rewards", "Reward", "Reward on Original Levels"),
    ]:
        ax = axes[panel]
        data = log[metric]
        for lv in range(1, 10):
            vals = [d.get(lv, d.get(str(lv), 0)) for d in data]
            if metric == "checkpoint_level_wins":
                vals = [v * 100 for v in vals]
            ax.plot(
                [s / 1e6 for s in ckpt_steps], vals,
                marker="o", markersize=4, label=CURRICULUM[lv]["name"],
                color=colors_lv[lv - 1], linewidth=1.5,
            )
        ax.set_xlabel("Timesteps (M)")
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.legend(fontsize=7, ncol=3, loc="upper left")
        ax.grid(alpha=0.3)

    plt.tight_layout()
    chart_path3 = OUTPUT_DIR / "alp_gmm_generalization_over_time.png"
    fig.savefig(chart_path3, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {chart_path3}")
else:
    print("Not enough checkpoints for progression chart.")

In [ ]:
# ── ALP-GMM behavior summary ─────────────────────────────────
print("=" * 60)
print("ALP-GMM SUMMARY")
print("=" * 60)
stats = alp_gmm.stats()
for k, v in stats.items():
    print(f"  {k}: {v}")

print(f"\n  Total training time: {elapsed/60:.1f} min")
print(f"  Total tasks sampled: {task_count}")
print(f"  Total timesteps: {cumulative_steps:,}")

if len(diffs) > 0:
    q1 = len(diffs) // 4
    q2 = len(diffs) // 2
    q3 = 3 * len(diffs) // 4
    print(f"\n  Difficulty progression (mean per quarter):")
    print(f"    Q1 (early):  {diffs[:q1].mean():.3f}")
    print(f"    Q2:          {diffs[q1:q2].mean():.3f}")
    print(f"    Q3:          {diffs[q2:q3].mean():.3f}")
    print(f"    Q4 (late):   {diffs[q3:].mean():.3f}")

print("\n  Final success rates on original levels:")
for lv in range(1, 10):
    r = final_results[lv]
    bar = "█" * int(r["success"] * 20)
    print(f"    Level {lv:>2d} ({r['name']:<12s}) {bar:<20s} {r['success']*100:.1f}%")

print("\nAll outputs saved to:", OUTPUT_DIR)

## Kết luận & So sánh

### So sánh Experiment 1 vs Experiment 2

| | Exp 1: Fixed Curriculum | Exp 2: ALP-GMM |
|---|---|---|
| **Cách chọn task** | Tuần tự 9 level cố định | Tự động, dựa trên learning progress |
| **Biên giới học** | Cố định bởi con người | Agent tự tìm |
| **Anti-forgetting** | Không (chỉ đi tiến) | Tự nhiên (GMM quay lại vùng suy giảm) |
| **Generalization** | Chỉ 9 cấu hình | Hàng trăm cấu hình đa dạng |
| **Hyperparameter** | Level order, mastery gates | ε, GMM components, steps_per_task |

### Output files
- `outputs/alp_gmm_ppo_final.zip` — model cuối cùng
- `outputs/alp_gmm_results.json` — toàn bộ metrics + log
- `outputs/alp_gmm_curriculum_analysis.png` — 4 biểu đồ phân tích chính
- `outputs/alp_gmm_training_progress.png` — reward & win rate
- `outputs/alp_gmm_generalization_over_time.png` — performance trên 9 level qua thời gian

### Gợi ý
- Chạy smoke-test: đặt `TIMESTEP_SCALE = 0.1` ở cell cấu hình
- So sánh kết quả với `outputs/experiment_all_models_results.json` từ Experiment 1